In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# 1. CARGA DE DATOS
# ---------------------------------------------------------
try: 
    contract = pd.read_csv (r'C:\Users\juand\OneDrive\Escritorio\TripleTen\Telecom\Data\contract.csv')
    personal = pd.read_csv (r'C:\Users\juand\OneDrive\Escritorio\TripleTen\Telecom\Data\personal.csv')
    internet = pd.read_csv (r'C:\Users\juand\OneDrive\Escritorio\TripleTen\Telecom\Data\internet.csv')
    phone = pd.read_csv (r'C:\Users\juand\OneDrive\Escritorio\TripleTen\Telecom\Data\phone.csv')
except FileNotFoundError:
    print ("❌ Error: Asegúrate de subir los archivos al entorno.")

# ---------------------------------------------------------
# 2. FUSIÓN (MERGE)
# ---------------------------------------------------------
df_merged = contract.merge (personal, on='customerID', how='left')
df_merged = df_merged.merge (internet, on='customerID', how='left')
df_merged = df_merged.merge (phone, on='customerID', how='left')

# ---------------------------------------------------------
# 3. LIMPIEZA DE NULOS Y TIPOS DE DATOS
# ---------------------------------------------------------

# A) Rellenar Nulos de servicios
# Los NaNs generados por el merge significan que el cliente no tiene el servicio
# Lista de columnas que vienen de las tablas 'internet' y 'phone'

cols_servicios = [
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 
    'StreamingMovies', 'MultipleLines'
]

# Rellenamos con 'No'
for col in cols_servicios: 
    df_merged[col] = df_merged[col].fillna('No')

# B) Corregir TotalCharges
df_merged['TotalCharges'] = pd.to_numeric(df_merged['TotalCharges'], errors='coerce')

# Imputamos los nulos (Clientes nuevos con tenure 0) con 0
df_merged['TotalCharges'] = df_merged['TotalCharges'].fillna(0)

# ---------------------------------------------------------
# 4. FEATURE ENGINEERING (Ingeniería de Características)
# ---------------------------------------------------------

# A) Crear variable objectivo 'Churn' (Target)
# 1 = Se fue (EndDate tiene fecha), 0 = Se quedó (EndDate es 'No')
df_merged['Churn'] = (df_merged['EndDate']!='No').astype(int)

# B) Crear variable 'Tenure' (Antigüedad en días)
# Fecha de referencia dada por el negocio: 1 de febrero de 2020
fecha_corte = pd.to_datetime ('2020-02-01')

# Convertir BeginDate a datetime
df_merged['BeginDate'] = pd.to_datetime (df_merged['BeginDate'])

# Lógica: Si EndDate es 'No', usamos la fecha de corte. Si tiene fecha usamos esa.
df_merged['EndDate_Calc'] = df_merged['EndDate'].replace('No', fecha_corte)
df_merged['EndDate_Calc'] = pd.to_datetime(df_merged['EndDate_Calc'])

# Calculamos los días de diferencia 
df_merged['Tenure_Days'] = (df_merged['EndDate_Calc'] - df_merged['BeginDate']).dt.days

# ---------------------------------------------------------
# 5. REVISIÓN FINAL
# ---------------------------------------------------------

print (f'Dimensiones finales: {df_merged.shape}')
print ('\n--- Vista previa de las nuevas columnas ---')
display(df_merged[['customerID', 'BeginDate', 'EndDate', 'Churn', 'Tenure_Days', 'TotalCharges']].head())

print("\n--- Info del Dataset Maestro ---")
df_merged.info()

Dimensiones finales: (7043, 23)

--- Vista previa de las nuevas columnas ---


,customerID,BeginDate,EndDate,Churn,Tenure_Days,TotalCharges
0,7590-VHVEG,2020-01-01,No,0,31,29.85
1,5575-GNVDE,2017-04-01,No,0,1036,1889.50
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,1,61,108.15
3,7795-CFOCW,2016-05-01,No,0,1371,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,1,61,151.65



--- Info del Dataset Maestro ---
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customerID        7043 non-null   str           
 1   BeginDate         7043 non-null   datetime64[us]
 2   EndDate           7043 non-null   str           
 3   Type              7043 non-null   str           
 4   PaperlessBilling  7043 non-null   str           
 5   PaymentMethod     7043 non-null   str           
 6   MonthlyCharges    7043 non-null   float64       
 7   TotalCharges      7043 non-null   float64       
 8   gender            7043 non-null   str           
 9   SeniorCitizen     7043 non-null   int64         
 10  Partner           7043 non-null   str           
 11  Dependents        7043 non-null   str           
 12  InternetService   7043 non-null   str           
 13  OnlineSecurity    7043 non-null   str           
 14  O